# 03 · NumPy 루틴 (cupy.* · linalg · fft · random)

> **CuPy 2일 집중 코스 — Day 1 / 단원 2 (NumPy/SciPy CuPy 프로그래밍)**

CuPy 공식 [overview](https://docs.cupy.dev/en/stable/overview.html)의 **NumPy Routines** 를 본격적으로 다룹니다.
모듈 함수·선형대수·FFT·난수를 예제와 **미니앱**(PCA, 2D 디노이즈, 몬테카를로)으로 익히고, 커널 퓨전까지 맛봅니다.

## 학습 목표
- `cupy.*` 모듈 함수(집계/정렬/탐색/누적/집합)를 자유롭게 쓴다.
- `cupy.linalg`로 PCA·배치 행렬연산을, `cupy.fft`로 2D 주파수 처리를 구현한다.
- `cupy.random`으로 몬테카를로 시뮬레이션을 GPU에서 수행한다.
- `@cupy.fuse`로 원소 연산을 융합해 temporary를 줄인다.

## 목차
1. [모듈 루틴 투어 `cupy.*`](#1)
2. [선형대수 `cupy.linalg` + PCA](#2)
3. [FFT `cupy.fft` + 2D 디노이즈](#3)
4. [난수 `cupy.random` + 몬테카를로](#4)
5. [커널 퓨전 `@cupy.fuse`](#5)
6. [체크포인트](#6)

> 백엔드: cuBLAS/cuSOLVER(linalg) · cuFFT(fft) · cuRAND(random). 전체 구성은 `00`의 *CuPy 루틴 지도* 참고.

In [ ]:
import os, sys, time, math
import numpy as np
import cupy as cp
from course_utils import print_env, bench, gpu_ms, cpu_ms, compare, allclose
print_env()

### (주의) CPU vs GPU 연산 결과가 미세하게 다른 이유

CPU와 GPU는 모두 **IEEE 754** 부동소수점 표준을 따릅니다. 32비트 Float을 부호(1bit) + 지수(8bit) + 가수(23bit)로 저장하는 방식도 동일합니다.

차이는 **병렬화 방식**에서 비롯됩니다.

- **CPU**: 최대 수십 개 코어로 병렬 연산
- **GPU**: 수천 개 코어로 극단적인 병렬 연산

행렬 곱셈이나 리덕션(Reduction, 총합 구하기 등) 연산에서는 **덧셈이 이루어지는 순서**가 CPU와 GPU에서 달라질 수 있습니다. 부동소수점 덧셈은 결합법칙이 항상 성립하지 않기 때문에, 순서가 달라지면 마지막 비트(LSB, Least Significant Bit)가 반올림되거나 버려지는 시점이 달라지고, 그 결과 **비트 단위의 미세한 차이**가 발생합니다.

> 즉, 하드웨어 표준(IEEE 754)이 같아도 연산 순서 차이 때문에 CPU와 GPU의 계산 결과가 완전히 동일하지 않을 수 있습니다.

<a id="1"></a>
## 1. 모듈 루틴 투어 `cupy.*`

📖 [`cupy.*` 루틴 전체](https://docs.cupy.dev/en/stable/reference/routines.html) — 대부분의 `numpy.*` 가 같은 이름으로 존재합니다.
집계·정렬·탐색·누적·집합·논리 함수를 한 번에 둘러봅니다.

In [ ]:
x_np = np.random.random(2_000_000).astype(np.float32)
x = cp.asarray(x_np)
print('mean/median :', float(x.mean()), float(cp.median(x)))
print('std/var     :', float(x.std()), float(x.var()))
print('percentile90:', float(cp.percentile(x, 90)))
print('argmax      :', int(cp.argmax(x)))
print('cumsum[-1]  :', float(cp.cumsum(x)[-1]))
u = cp.unique(cp.asarray([3,1,2,3,1]))
print('unique      :', cp.asnumpy(u))
print('searchsorted:', int(cp.searchsorted(cp.sort(x), 0.5)))

**연습 1 — Top-k**: 가장 큰 `k`개 값을 내림차순으로 반환하는 장치 비종속 함수를 완성하세요.

In [ ]:
def topk(x, k):
    # TODO: xp.sort(x)[-k:][::-1] 또는 argpartition 사용
    raise NotImplementedError

x_np = np.random.randn(1_000_000).astype(np.float32)
# ref = np.sort(x_np)[-5:][::-1]; out = cp.asnumpy(topk(cp.asarray(x_np), 5))
# "CPU 결과와 GPU 결과가 똑같은지" 검증하는 코드
# allclose(ref, out, name='topk')

<details><summary>💡 해답 보기</summary>

```python
def topk(x, k):
    xp = cp.get_array_module(x)
    return xp.sort(x)[-k:][::-1]

x_np = np.random.randn(1_000_000).astype(np.float32)
ref = np.sort(x_np)[-5:][::-1]
out = cp.asnumpy(topk(cp.asarray(x_np), 5))
allclose(ref, out, name='topk')
```
</details>

**연습 2 — 이동평균(cumsum 트릭)**: `cumsum`으로 길이 `w` 이동평균을 O(N)에 구하세요.

In [ ]:
def moving_avg(x, w):
    # TODO: c = xp.cumsum(x); (c[w:]-c[:-w])/w  (앞부분 처리 포함)
    raise NotImplementedError

x_np = np.arange(20, dtype=np.float32)
# print(cp.asnumpy(moving_avg(cp.asarray(x_np), 5)))

<details><summary>💡 해답 보기</summary>

```python
def moving_avg(x, w):
    xp = cp.get_array_module(x)
    c = xp.cumsum(x)
    return (c[w-1:] - xp.concatenate([xp.zeros(1, x.dtype), c[:-w]])) / w

x_np = np.arange(20, dtype=np.float32)
print(cp.asnumpy(moving_avg(cp.asarray(x_np), 5)))
```
</details>

<a id="2"></a>
## 2. 선형대수 `cupy.linalg` + PCA

📖 [`cupy.linalg`](https://docs.cupy.dev/en/stable/reference/linalg.html) — `solve`, `svd`, `eigh`, `qr`, `inv`, `norm`, `lstsq` 등 (cuSOLVER/cuBLAS)

### 🔬 이론 배경 — 선형대수 & PCA

1500차원의 거대한 무작위 행렬을 만들어서, 
1) CPU와 GPU가 똑같이 연립방정식을 잘 푸나 테스트(solve)해보고, 
2) GPU로 행렬의 특이값(Singular Values)을 뽑아내어 
3) 이 행렬이 얼마나 안정적인 구조인지 확인(cond)하는 벤치마크 및 검증 코드 예제입니다.

- **solve(Ax=b)**: A를 LU로 분해해 삼각계 후진대입으로 해를 구함(대략 O(n³)).
- **SVD** `A = U Σ Vᵀ`: 특이값 Σ가 데이터의 '주축 크기' → 차원축소·저랭크 근사.
- **eigh**(대칭 고유분해)로 **PCA**: 공분산행렬의 **고유벡터 = 분산이 최대인 방향**, 고유값 = 그 방향의 분산량.

In [ ]:
# 기본기: solve / svd / eigh / norm
n = 1500
A_np = np.random.randn(n, n).astype(np.float32); b_np = np.random.randn(n).astype(np.float32)
A_cp, b_cp = cp.asarray(A_np), cp.asarray(b_np)
allclose(np.linalg.solve(A_np, b_np), cp.linalg.solve(A_cp, b_cp), rtol=2e-3, atol=2e-3, name='solve')
S = cp.linalg.svd(A_cp, compute_uv=False)
print('cond(A)~', float(S[0]/S[-1]), '| ||A||_2 =', float(S[0]))

**배치 행렬곱**: 스택된 `(B, n, n)` 행렬을 한 번에 곱하면(`cp.matmul`) GPU가 특히 강합니다.

In [ ]:
Bm, n = 64, 256
A = cp.random.random((Bm, n, n), dtype=cp.float32)
Bb = cp.random.random((Bm, n, n), dtype=cp.float32)
Cb = cp.matmul(A, Bb)              # 배치 matmul
print('batched matmul out:', Cb.shape)
A_np = cp.asnumpy(A); B_np = cp.asnumpy(Bb)
compare('bmm', lambda: np.matmul(A_np, B_np), lambda: cp.matmul(A, Bb), n_repeat=5, n_warmup=2)

### 미니앱 — PCA (주성분 분석)
공분산 행렬의 고유분해(`eigh`)로 주성분을 구합니다. (고유벡터는 부호 모호성이 있어 **고유값**으로 검증)

In [ ]:
def pca(X, k):
    # -------------------------------------------------------------------------
    # 1. 중심화 (Centering)
    #    - 각 열(특징)의 평균을 구해서 X에서 빼줍니다.
    #    - 힌트: X.mean(axis=0, keepdims=True)을 활용해 브로드캐스팅(차원 맞춤) 하세요.
    #    - Shape: (N, D) -> (N, D)
    #
    # 2. 공분산 행렬 계산 (Covariance Matrix)
    #    - 공식: C = (Xc.T @ Xc) / (N - 1)
    #    - 힌트: N은 데이터의 개수(X.shape[0])입니다.
    #    - Shape: (D, N) @ (N, D) -> (D, D)
    #
    # 3. 고유분해 (Eigen Decomposition)
    #    - 함수: xp.linalg.eigh(C)  (xp는 np 또는 cp)
    #    - 반환값: w(고유값 배열), V(고유벡터 행렬)
    #    - 주의: eigh는 '오름차순(작은 값->큰 값)'으로 정렬되어 나옵니다!
    #
    # 4. 상위 k개 고유값 및 주성분 추출 (Sorting & Slicing)
    #    - 힌트: 내림차순(큰 값->작은 값)으로 뒤집은 뒤, 앞의 k개만 잘라내야 합니다.
    #    - 배열 뒤집기 팁: xp.argsort(w)[::-1][:k]
    #    - comps (주성분) Shape: V에서 해당 인덱스의 열만 선택 -> (D, k)
    #
    # 5. 차원 축소 투영 (Projection)
    #    - 공식: Xc @ comps
    #    - Shape: (N, D) @ (D, k) -> (N, k)
    # -------------------------------------------------------------------------

    # TODO: 중심화 -> 공분산 C=(Xc.T@Xc)/(N-1) -> eigh -> 상위 k 고유값/주성분 -> 투영(Xc@comps)
    #       반환: (투영 (N,k), 상위 k 고유값 내림차순)
    raise NotImplementedError

X_np = (np.random.randn(5000, 20) @ np.random.randn(20, 20)).astype(np.float32)
# _, val_np = pca(X_np, 5)
# _, val_cp = pca(cp.asarray(X_np), 5)
# allclose(val_np, cp.asnumpy(val_cp), rtol=1e-2, atol=1e-2, name='PCA eigenvalues')

<details><summary>💡 해답 보기</summary>

```python
def pca(X, k):
    xp = cp.get_array_module(X)
    Xc = X - X.mean(axis=0, keepdims=True)
    Cov = (Xc.T @ Xc) / (X.shape[0] - 1)
    w, V = xp.linalg.eigh(Cov)            # 오름차순
    idx = xp.argsort(w)[::-1][:k]
    comps = V[:, idx]
    return Xc @ comps, w[idx]

X_np = (np.random.randn(5000, 20) @ np.random.randn(20, 20)).astype(np.float32)
_, val_np = pca(X_np, 5)
_, val_cp = pca(cp.asarray(X_np), 5)
allclose(val_np, cp.asnumpy(val_cp), rtol=1e-2, atol=1e-2, name='PCA eigenvalues')
```
</details>

**연습 — 최소제곱(lstsq)**: 과결정계 `Ax≈b`를 `linalg.lstsq`로 푸세요.

1. 문제 정의
- `A`: (2000, 50) 행렬 → **방정식 2000개, 미지수 50개**
- `b`: (2000,) 벡터

방정식 개수(2000)가 미지수 개수(50)보다 훨씬 많은 **과결정계(overdetermined system)**입니다. 이런 경우 보통 `Ax = b`를 정확히 만족하는 해가 존재하지 않기 때문에, 오차를 최소화하는 근사해를 구해야 합니다.

2. 최소제곱해 (Least Squares Solution)
`xp.linalg.lstsq(A, b, rcond=None)[0]`는 다음을 최소화하는 `x`를 찾습니다.

$$
\min_x \; \| Ax - b \|_2^2
$$

내부적으로 SVD(특이값 분해)나 QR 분해를 이용해 계산하며, `rcond=None`은 특이값 중 무시할 정도로 작은 값(rank 판정 기준)을 라이브러리 기본값(머신 epsilon 기반)으로 설정하는 옵션입니다 (경고 메시지 방지 목적).


In [ ]:
def solve_lstsq(A, b):
    # TODO: xp.linalg.lstsq(A, b, rcond=None)[0]
    raise NotImplementedError
# A=np.random.randn(2000,50).astype('f4'); b=np.random.randn(2000).astype('f4')
# allclose(solve_lstsq(A,b), cp.asnumpy(solve_lstsq(cp.asarray(A),cp.asarray(b))), rtol=2e-2, atol=2e-2, name='lstsq')

<details><summary>💡 해답 보기</summary>

```python
def solve_lstsq(A, b):
    xp = cp.get_array_module(A)
    return xp.linalg.lstsq(A, b, rcond=None)[0]
```
</details>

<a id="3"></a>
## 3. FFT `cupy.fft` + 2D 디노이즈

📖 [`cupy.fft`](https://docs.cupy.dev/en/stable/reference/fft.html) — `fft/ifft`, `rfft/irfft`, `fft2/ifft2`, `fftfreq`, `fftshift`

### 🔬 이론 배경 — FFT가 하는 일
- **DFT**는 신호를 주파수 성분으로 분해: `X_k = Σ_n x_n · e^(−2πi·kn/N)`.
- 직접 계산은 O(N²), **FFT**는 분할정복으로 **O(N log N)**.
- `rfft`는 실수 입력의 대칭성을 이용해 계산·저장을 절반으로.
- **컨볼루션 정리**: 시간영역 컨볼루션 = 주파수영역 **곱** → 큰 필터를 FFT로 빠르게 적용.

In [ ]:
# 1D: rfft/irfft 로 FFT 컨볼루션 (장치 비종속)
def fftconv(x, h):
    xp = cp.get_array_module(x); n = x.size + h.size - 1
    return xp.fft.irfft(xp.fft.rfft(x, n) * xp.fft.rfft(h, n), n)
x_np = np.random.randn(500_000).astype(np.float32); h_np = np.exp(-np.linspace(0,8,2048)).astype(np.float32)
allclose(fftconv(x_np,h_np), fftconv(cp.asarray(x_np),cp.asarray(h_np)), rtol=2e-3, atol=2e-3, name='fftconv')

### 미니앱 — 2D 주파수 저역통과 디노이즈
이미지를 `fft2`→`fftshift` 후 중앙(저주파) 사각형만 남기고 역변환하면 고주파 잡음이 제거됩니다.

작동 원리:

1. 이미지를 주파수 세상으로 보냅니다 (FFT 변환).
2. 주파수 세상의 가운데(주파수가 0인 부근)에는 '부드러운 성분(저주파)'이 모여 있고, 바깥쪽에는 '거칠고 자글자글한 성분(고주파 잡음)'이 모여 있습니다.
3. 가운데 중심에서 'radius(반지름)' 크기만큼의 사각형만 남기고 바깥쪽은 0으로 다 지웁니다 (마스크 적용).
4. 깨끗해진 주파수를 다시 원래 이미지 세상으로 되돌립니다 (IFFT 역변환).

In [ ]:
def lowpass2d(img, r):
    # -------------------------------------------------------------------------
    # Step 1: 이미지를 주파수 공간으로 변환하고, 중심이 가운데로 오도록 셔플(shift)합니다.
    # -> Shape 변환 없음: (H, W) 복소수 행렬
    # Step 2: 중심(저주파)만 통과시킬 검은색 사각형 필터(초기값 0)를 만듭니다.
    #   h, w = img.shape; cy, cx = h//2, w//2
    #   m = xp.zeros((h, w), dtype=img.dtype)
    # Step 3: 정가운데 영역에만 흰색 네모(1)를 그립니다. (이 안의 주파수만 살아남음)
    #   m[cy-r:cy+r, cx-r:cx+r] = 1   
    # Step 4: 주파수 정보와 마스크를 곱해 바깥쪽 잡음을 날려버립니다.
    # Step 5: 역변환을 통해 다시 눈에 보이는 이미지로 되돌린 후, 실수(Real) 값만 취합니다.
    # -> 힌트: xp.fft.ifft2()와 xp.fft.ifftshift()를 적절히 조합하세요.
 
    # TODO: F=fftshift(fft2(img)); 중앙 (2r x 2r)만 남기고 0; real(ifft2(ifftshift(F)))
    raise NotImplementedError

rng = np.random.default_rng(0)
yy, xx = np.mgrid[0:128, 0:128]
img_np = (np.sin(xx/8.0) + 0.6*rng.standard_normal((128,128))).astype(np.float32)
# out_np = lowpass2d(img_np, 12); out_cp = cp.asnumpy(lowpass2d(cp.asarray(img_np), 12))
# allclose(out_np, out_cp, rtol=2e-3, atol=2e-3, name='lowpass2d')

<details><summary>💡 해답 보기</summary>

```python
def lowpass2d(img, r):
    xp = cp.get_array_module(img)
    F = xp.fft.fftshift(xp.fft.fft2(img))
    h, w = img.shape; cy, cx = h//2, w//2
    m = xp.zeros((h, w), dtype=img.dtype)
    m[cy-r:cy+r, cx-r:cx+r] = 1
    return xp.real(xp.fft.ifft2(xp.fft.ifftshift(F * m)))

out_np = lowpass2d(img_np, 12)
out_cp = cp.asnumpy(lowpass2d(cp.asarray(img_np), 12))
allclose(out_np, out_cp, rtol=2e-3, atol=2e-3, name='lowpass2d')
```
</details>

In [ ]:
# 시각화
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(img_np, cmap='gray')
plt.title("1. Before (Original + Noise)")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(out_np, cmap='gray')
plt.title("2. After (Lowpass, r=12)")
plt.axis('off')

plt.tight_layout()
plt.show()

<a id="4"></a>
## 4. 난수 `cupy.random` + 몬테카를로

📖 [`cupy.random`](https://docs.cupy.dev/en/stable/reference/random.html) — 권장 **Generator API** `cp.random.default_rng()` (cuRAND)

### 🔬 이론 배경 — 난수 & 몬테카를로
- **PRNG**: 시드로 결정되는 의사난수열(재현 가능, 참난수 아님).
- **몬테카를로**: 난수 표본으로 적분·확률을 추정. 표준오차 ~ **1/√N**(표본 수 N).
- 정확도를 높이려면 표본이 많이 필요 → **대량 난수 생성에 GPU가 유리**.

In [ ]:
rng = cp.random.default_rng(0)
print('normal :', cp.asnumpy(rng.standard_normal(3, dtype=cp.float32)))
print('int    :', cp.asnumpy(rng.integers(0, 10, size=5)))
print('choice :', cp.asnumpy(cp.random.choice(cp.arange(100), size=5, replace=False))) 
# 재현성: 같은 시드 -> 같은 수열 (같은 버전/디바이스)
a = cp.random.default_rng(1).standard_normal(4, dtype=cp.float32)
b = cp.random.default_rng(1).standard_normal(4, dtype=cp.float32)
print('재현?', bool((a==b).all()))

### 미니앱 — 몬테카를로로 π 추정
대량 난수는 GPU의 강점입니다. 단위정사각형에 점을 뿌려 1/4원 내부 비율로 π를 추정합니다.

In [ ]:
def mc_pi(n, seed=0):
    # TODO: rng=cp.random.default_rng(seed); x,y 난수; inside=(x*x+y*y<=1).sum(); 4*inside/n
    raise NotImplementedError

# for n in [10**5, 10**6, 10**7, 10**8]:
#     est = mc_pi(n); print(f'N={n:>10,}  π~{est:.5f}  err={abs(est-math.pi):.2e}')

<details><summary>💡 해답 보기</summary>

```python
def mc_pi(n, seed=0):
    rng = cp.random.default_rng(seed)
    x = rng.random(n, dtype=cp.float32); y = rng.random(n, dtype=cp.float32)
    inside = ((x*x + y*y) <= 1.0).sum()
    return float(4 * inside / n)

for n in [10**5, 10**6, 10**7, 10**8]:
    est = mc_pi(n)
    print(f'N={n:>11,}  pi~{est:.5f}  err={abs(est-math.pi):.2e}')
# 오차가 ~1/sqrt(N) 로 줄어드는지 관찰하세요.
```
</details>

<a id="5"></a>
## 5. 커널 퓨전 `@cupy.fuse`

📖 [`cupy.fuse`](https://docs.cupy.dev/en/stable/reference/generated/cupy.fuse.html) — 여러 원소/리덕션 연산을 **하나의 커널로 융합**해 중간배열(temporary)과 커널 런치를 줄입니다.
처음 호출 시 dtype/ndim에 맞춰 커널을 컴파일·캐시하므로, **같은 데코레이트 함수를 재사용**하세요. (Day 2 커널 작성의 예고편)

1. a*a, b*b, +. log1p 연산에 대해서 4개의 커널을 순차적으로 실행, 커널 수행 오버헤드가 큼 

```python
def elementwise_plain(a, b, c):
    return cp.sqrt(a*a + b*b) + cp.log1p(c)
```

2. a*a, b*b, +. log1p 연산에 대해서 4개의 커널을 한번에 모아서 실행

```python
@cp.fuse()
def elementwise_fused(a, b, c):
    return cp.sqrt(a*a + b*b) + cp.log1p(c)
```

In [ ]:
def elementwise_plain(a, b, c):
    return cp.sqrt(a*a + b*b) + cp.log1p(c)

@cp.fuse()
def elementwise_fused(a, b, c):
    return cp.sqrt(a*a + b*b) + cp.log1p(c)

a = cp.random.random(30_000_000, dtype=cp.float32)
b = cp.random.random(30_000_000, dtype=cp.float32)
c = cp.random.random(30_000_000, dtype=cp.float32)
_ = elementwise_fused(a, b, c)   # 워밍업(첫 호출에 커널 컴파일)
r1 = bench(lambda: elementwise_plain(a,b,c), n_repeat=20, name='plain')
r2 = bench(lambda: elementwise_fused(a,b,c), n_repeat=20, name='fused')
print(f'plain {gpu_ms(r1):.3f} ms | fused {gpu_ms(r2):.3f} ms | speedup {gpu_ms(r1)/gpu_ms(r2):.2f}x')

**연습 — 함수 융합**: 아래 `f`를 `@cp.fuse()`로 융합하고 정확성·속도를 확인하세요.

In [ ]:
def f(x, y):
    return (cp.tanh(x) + 1.0) * cp.exp(-y*y)

# TODO: @cp.fuse() 로 f_fused 정의 후 동일 결과·속도 비교

#x = cp.random.random(30_000_000, dtype=cp.float32)
#y = cp.random.random(30_000_000, dtype=cp.float32)
#allclose(f(x,y), f_fused(x,y), rtol=1e-5, atol=1e-5, name='fuse')
#_ = f_fused(x,y)
#print('plain', gpu_ms(bench(lambda: f(x,y))), 'ms | fused', gpu_ms(bench(lambda: f_fused(x,y))), 'ms')

<details><summary>💡 해답 보기</summary>

```python
def f(x, y):
    return (cp.tanh(x) + 1.0) * cp.exp(-y*y)

@cp.fuse()
def f_fused(x, y):
    return (cp.tanh(x) + 1.0) * cp.exp(-y*y)

x = cp.random.random(30_000_000, dtype=cp.float32)
y = cp.random.random(30_000_000, dtype=cp.float32)
allclose(f(x,y), f_fused(x,y), rtol=1e-5, atol=1e-5, name='fuse')
_ = f_fused(x,y)
print('plain', gpu_ms(bench(lambda: f(x,y))), 'ms | fused', gpu_ms(bench(lambda: f_fused(x,y))), 'ms')
```
</details>

<a id="6"></a>
## 6. 체크포인트

- [ ] `cupy.*` 모듈 함수(집계/정렬/탐색/누적/집합)를 사용했다
- [ ] `cupy.linalg`로 solve/svd/eigh와 **PCA**를 구현했다
- [ ] 배치 `matmul`로 GPU 강점을 확인했다
- [ ] `cupy.fft`로 1D 컨볼루션과 **2D 디노이즈**를 했다
- [ ] `cupy.random`으로 **몬테카를로 π**를 추정했다(오차~1/√N)
- [ ] `@cupy.fuse`로 원소 연산을 융합해 속도를 높였다

다음: **`04_scipy_routines`** — SciPy 루틴(fft·linalg·ndimage·sparse·signal·…).